# 05 — FastAPI Demo
**CLPapersAIAgent | Member 4**

Demonstrates the `/search` endpoint of the Paper Search API.
- Server: `http://127.0.0.1:8000`
- Methods: BM25, Dense, Hybrid
- Shows example queries with top-k results and citations

In [4]:

import requests
import pandas as pd

BASE = "http://127.0.0.1:8000"


print(requests.get(f"{BASE}/").json())

{'message': 'Paper Search API is running', 'output_dir': 'C:\\Users\\97150\\Documents\\GitHub\\CLPapersAIAgent\\outputs\\test4', 'chunks_loaded': 6728, 'collection': 'clpapers_chunks'}


In [5]:
#  Example queries across all 3 methods
queries = [
    "retrieval augmented generation",
    "transformer architecture for sequence tasks",
    "large language model alignment"
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    for method in ["bm25", "dense", "hybrid"]:
        resp = requests.get(f"{BASE}/search", params={
            "query": query, "top_k": 3, "method": method
        }).json()
        print(f"\n  [{method.upper()}] — {resp['latency_ms']} ms")
        for r in resp["results"]:
            print(f"    #{r['rank']} {r['title'][:60]} (score: {r['score']})")


QUERY: retrieval augmented generation

  [BM25] — 20.17 ms
    #1 H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retr (score: 13.2)
    #2 Generation: A Controlled Empirical Study1 (score: 12.7714)
    #3 Generation: A Controlled Empirical Study1 (score: 12.6298)

  [DENSE] — 94.0 ms
    #1 FT-RAG: A Fine-grained Retrieval-Augmented Generation Framew (score: 0.6336)
    #2 Enhancing Judgment Document Generation via Agentic Legal Inf (score: 0.6145)
    #3 Beyond Semantic Relevance: Counterfactual Risk Minimization  (score: 0.6132)

  [HYBRID] — 55.02 ms
    #1 H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retr (score: 0.7697)
    #2 Generation: A Controlled Empirical Study1 (score: 0.7662)
    #3 Generation: A Controlled Empirical Study1 (score: 0.7654)

QUERY: transformer architecture for sequence tasks

  [BM25] — 27.51 ms
    #1 Computational Job Market Analysis (score: 18.763)
    #2 Reservoir Computing inspired Matrix Multiplication-free Lang (score: 16.0569)
 

In [6]:
# Cell 4 — Clean results table
rows = []
for query in queries:
    for method in ["bm25", "dense", "hybrid"]:
        resp = requests.get(f"{BASE}/search", params={
            "query": query, "top_k": 3, "method": method
        }).json()
        for r in resp["results"]:
            rows.append({
                "query":      query[:40],
                "method":     method,
                "rank":       r["rank"],
                "title":      r["title"][:50],
                "score":      r["score"],
                "latency_ms": resp["latency_ms"]
            })

pd.DataFrame(rows)

,query,method,rank,title,score,latency_ms
0,retrieval augmented generation,bm25,1,H-RAG at SemEval-2026 Task 8: Hierarchical Par...,13.2000,20.01
1,retrieval augmented generation,bm25,2,Generation: A Controlled Empirical Study1,12.7714,20.01
2,retrieval augmented generation,bm25,3,Generation: A Controlled Empirical Study1,12.6298,20.01
3,retrieval augmented generation,dense,1,FT-RAG: A Fine-grained Retrieval-Augmented Gen...,0.6336,45.52
4,retrieval augmented generation,dense,2,Enhancing Judgment Document Generation via Age...,0.6145,45.52
5,retrieval augmented generation,dense,3,Beyond Semantic Relevance: Counterfactual Risk...,0.6132,45.52
6,retrieval augmented generation,hybrid,1,H-RAG at SemEval-2026 Task 8: Hierarchical Par...,0.7697,53.54
7,retrieval augmented generation,hybrid,2,Generation: A Controlled Empirical Study1,0.7662,53.54
8,retrieval augmented generation,hybrid,3,Generation: A Controlled Empirical Study1,0.7654,53.54
9,transformer architecture for sequence ta,bm25,1,Computational Job Market Analysis,18.7630,30.51


In [8]:
print("\n" + "="*60)
print("TOP-K RESULTS WITH CITATIONS")
print("="*60)

queries = [
    "retrieval augmented generation",
    "transformer architecture for sequence tasks",
    "large language model alignment"
]

for query in queries:
    for method in ["bm25", "dense", "hybrid"]:
        resp = requests.get(f"{BASE}/search", params={
            "query": query,
            "top_k": 3,
            "method": method
        }).json()

        print(f"\n{'─'*60}")
        print(f"Query  : {resp['query']}")
        print(f"Method : {resp['method'].upper()} | Latency: {resp['latency_ms']}ms")
        print(f"{'─'*60}")

        for r in resp["results"]:
            print(f"  #{r['rank']} {r['title'][:70]}")
            print(f"       Authors : {r['authors'][:80]}")
            print(f"       Year    : {r['year']}")
            print(f"       Score   : {r['score']}")
            print(f"       Snippet : {r['snippet'][:150]}...")
            print()


TOP-K RESULTS WITH CITATIONS

────────────────────────────────────────────────────────────
Query  : retrieval augmented generation
Method : BM25 | Latency: 19.52ms
────────────────────────────────────────────────────────────
  #1 H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for 
       Authors : Passant Elchafei; Hossam Emam; Mohamed Alansary; Monorama Swain; Markus Schedl
       Year    : 2026
       Score   : 13.2
       Snippet : long-context dialogue systems. References Yiruo Cheng, Kelong Mao, Ziliang Zhao, Guanting Dong, Hongjin Qian, Yongkang Wu, Tetsuya Sakai, Ji-Rong Wen,...

  #2 Generation: A Controlled Empirical Study1
       Authors : Benchmarking Retrieval Strategies for Biomedical RAG; Benchmarking Retrieval Str
       Year    : unknown
       Score   : 12.7714
       Snippet : on Empirical Methods in Natural Language Processing (EMNLP), pp. 6769–6781, 2020. [8] G. Izacard et al., "Unsupervised Dense Information Retrieval wit...

  #3 Generation: A 